In [23]:
import os 
os.environ['AWS_ACCESS_KEY_ID'] = '<paste your access key here>'
os.environ['AWS_SECRET_ACCESS_KEY'] = '<paste your secret access key here'
os.environ['AWS_DEFAULT_REGION'] = 'us-east-1'

In [24]:
import boto3
import json

bedrock = boto3.client(
    service_name='bedrock-runtime',
    region_name='us-east-1'
)

print("🚀 Testing Bedrock with Amazon Titan...\n")

request_body = {
    "inputText": "Explain Amazon Bedrock in one sentence.",
    "textGenerationConfig": {
        "maxTokenCount": 512,
        "temperature": 0.7,
        "topP": 0.9
    }
}

try:
    response = bedrock.invoke_model(
        modelId='amazon.titan-text-express-v1',  # Using Titan instead
        contentType='application/json',
        accept='application/json',
        body=json.dumps(request_body)
    )
    
    response_body = json.loads(response['body'].read())
    answer = response_body['results'][0]['outputText']
    
    print(f"✅ SUCCESS!\n")
    print(f"Titan says: {answer}\n")
    
except Exception as e:
    print(f"❌ Error: {e}")
    print("\nThis model also requires payment method.")
    print("Please add a payment card to your AWS account.")

🚀 Testing Bedrock with Amazon Titan...

✅ SUCCESS!

Titan says: 
Amazon Bedrock is an AI-powered service that makes foundation models from leading AI startup and Amazon's own Titan models available through APIs. For example, you can use Amazon Bedrock to add image recognition to your mobile application, or to process language in real time.



In [25]:
import boto3
import json


class TitanChatbot:
    def __init__(self):
        """Initialize chatbot with Bedrock client"""
        self.bedrock = boto3.client(
            service_name='bedrock-runtime',
            region_name='us-east-1'
        )
        self.conversation_history = []
        self.total_tokens = 0
    
    def chat(self, user_message):
        """
        Send a message and get response from Titan
        """
        # Build conversation context
        # Titan doesn't have native conversation history like Claude
        # So we build it into a single prompt
        conversation_text = self._build_conversation_context()
        conversation_text += f"\nUser: {user_message}\nAssistant:"
        
        # Prepare request for Titan
        request_body = {
            "inputText": conversation_text,
            "textGenerationConfig": {
                "maxTokenCount": 1024,
                "temperature": 0.7,
                "topP": 0.9
                # Note: stopSequences removed - causes validation error
            }
        }
        
        try:
            # Call Bedrock with Titan
            response = self.bedrock.invoke_model(
                modelId='amazon.titan-text-express-v1',
                accept='application/json',
                contentType='application/json',
                body=json.dumps(request_body)
            )
            
            # Parse response
            response_body = json.loads(response['body'].read())
            assistant_message = response_body['results'][0]['outputText'].strip()
            
            # Track tokens for cost calculation
            input_tokens = response_body.get('inputTextTokenCount', 0)
            self.total_tokens += input_tokens
            
            # Store in history
            self.conversation_history.append({
                "user": user_message,
                "assistant": assistant_message
            })
            
            # Show token usage
            cost = self._calculate_cost(input_tokens)
            print(f"\n💰 [Input tokens: {input_tokens}, Cost: ${cost:.6f}]")
            
            return assistant_message
            
        except Exception as e:
            return f"Error: {str(e)}"
    
    def _build_conversation_context(self):
        """Build conversation history into a single text prompt"""
        if not self.conversation_history:
            return "You are a helpful AI assistant. Answer questions clearly and concisely."
        
        context = "You are a helpful AI assistant. Here's the conversation so far:\n\n"
        for turn in self.conversation_history[-5:]:  # Keep last 5 turns for context
            context += f"User: {turn['user']}\n"
            context += f"Assistant: {turn['assistant']}\n"
        
        return context
    
    def _calculate_cost(self, tokens):
        """Calculate cost for Titan (cheaper than Claude!)"""
        # Titan Text Express pricing: ~$0.20 per 1M input tokens, ~$0.60 per 1M output
        # This is approximate - check AWS pricing for exact rates
        return (tokens / 1_000_000) * 0.20
    
    def get_total_cost(self):
        """Get total cost of conversation"""
        return (self.total_tokens / 1_000_000) * 0.20
    
    def reset(self):
        """Clear conversation history"""
        total_cost = self.get_total_cost()
        print(f"\n📊 Session Summary:")
        print(f"   Total tokens: {self.total_tokens}")
        print(f"   Total cost: ${total_cost:.6f}")
        
        self.conversation_history = []
        self.total_tokens = 0
        print("🔄 Conversation cleared!\n")


def main():
    """
    Interactive chatbot loop
    """
    print("=" * 60)
    print("🤖 Amazon Titan Chatbot")
    print("=" * 60)
    print("\n✨ Using Amazon Titan Text Express (AWS's own LLM)")
    print("💰 More affordable than Claude!")
    print("\nCommands:")
    print("  - Type your message and press Enter")
    print("  - Type 'reset' to clear conversation")
    print("  - Type 'cost' to see total cost")
    print("  - Type 'quit' to exit")
    print("\n" + "=" * 60 + "\n")
    
    # Initialize chatbot
    chatbot = TitanChatbot()
    
    while True:
        # Get user input
        user_input = input("You: ").strip()
        
        # Handle commands
        if user_input.lower() == 'quit':
            total_cost = chatbot.get_total_cost()
            print(f"\n📊 Final Summary:")
            print(f"   Total cost: ${total_cost:.6f}")
            print("\n👋 Goodbye!\n")
            break
        
        if user_input.lower() == 'reset':
            chatbot.reset()
            continue
        
        if user_input.lower() == 'cost':
            total_cost = chatbot.get_total_cost()
            print(f"\n📊 Current Session:")
            print(f"   Total tokens: {chatbot.total_tokens}")
            print(f"   Total cost: ${total_cost:.6f}\n")
            continue
        
        if not user_input:
            continue
        
        # Get response
        print("\nTitan: ", end="", flush=True)
        response = chatbot.chat(user_input)
        print(response + "\n")


if __name__ == "__main__":
    main()

🤖 Amazon Titan Chatbot

✨ Using Amazon Titan Text Express (AWS's own LLM)
💰 More affordable than Claude!

Commands:
  - Type your message and press Enter
  - Type 'reset' to clear conversation
  - Type 'cost' to see total cost
  - Type 'quit' to exit




You:  what is amazon bedrock



Titan: 
💰 [Input tokens: 26, Cost: $0.000005]
Amazon Bedrock is a managed service that makes foundation models from leading AI startup and Amazon's own Titan models available through APIs. For example, you can use Amazon Bedrock to add image classification capabilities to your mobile application or use Amazon Bedrock to add natural language processing capabilities to your chatbot. By default, no customer data is used for service or model improvements, nor is it shared with any third party model providers. Learn more ({URL}).



You:  explain in simple terms 



Titan: 
💰 [Input tokens: 125, Cost: $0.000025]
Amazon Bedrock is a service that provides access to large language models and computer vision models. These models can be used to develop applications and services that can understand and process human language, images, and other data. For example, Amazon Bedrock can be used to build a chatbot that can understand and respond to customer questions, or to build an image recognition app that can identify objects in photos.

The service is available through APIs, which allow developers to integrate it into their applications and services. Amazon Bedrock also offers a set of tools and services that can help developers build and deploy their applications and services more quickly and efficiently.

In addition to its core capabilities, Amazon Bedrock also offers a range of services that can help developers improve the performance and scalability of their applications and services. For example, Amazon Bedrock can be used to deploy applications an

...


Titan: 
💰 [Input tokens: 385, Cost: $0.000077]
Amazon Bedrock is a managed service that provides access to large language models and computer vision models. These models can be used to develop applications and services that can understand and process human language, images, and other data. Amazon Bedrock is available through APIs, and it offers a range of tools and services that can help developers build and deploy their applications and services more quickly and efficiently. It is a powerful tool that can help developers build and deploy applications and services that can interact with humans and understand and process data more effectively.



You:  cost



📊 Current Session:
   Total tokens: 536
   Total cost: $0.000107



You:  quit



📊 Final Summary:
   Total cost: $0.000107

👋 Goodbye!

